In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
rent = pd.read_csv('/content/drive/MyDrive/Dataset/Origin/rent_dataset.csv', encoding='cp949')
print(rent.columns)

Index(['접수연도', '자치구코드', '자치구명', '법정동코드', '법정동명', '지번구분', '지번구분명', '본번', '부번',
       '층', '계약일', '전월세 구분', '임대면적(㎡)', '보증금(만원)', '임대료(만원)', '건물명', '건축년도',
       '건물용도', '계약기간', '신규갱신여부', '계약갱신권사용여부', '종전 보증금', '종전 임대료'],
      dtype='object')


In [ ]:
dong = pd.read_excel('/content/drive/MyDrive/Dataset/Origin/beopjeongdong_dataset.xlsx')
print(dong.columns)

Index(['시도명', '시군구명', '행정동명', '법정동명', '행정구역코드', '행정동코드', '법정동코드', '개정일자',
       '연결번호'],
      dtype='object')


In [ ]:
mapping = dong[['법정동명','행정동명']].copy()

In [ ]:
rent['법정동명'] = (rent['법정동명'].astype(str).str.strip())
mapping['법정동명'] = (mapping['법정동명'].astype(str).str.strip())
mapping['행정동명'] = (mapping['행정동명'].astype(str).str.strip())

In [ ]:
#매핑 예외 경우 확인
mapping_count = (
    mapping.groupby('법정동명')['행정동명']
    .nunique()
    .reset_index()
)

exception_case = mapping_count[
    mapping_count['행정동명'] > 1
]

total = len(mapping_count)
exception_num = len(exception_case)
ratio = exception_num / total * 100

print(f"전체 법정동: {total}")
print(f"1:N 매핑: {exception_num}")
print(f"비율: {ratio:.2f}%")

exception_case.head()

전체 법정동: 491
1:N 매핑: 137
비율: 27.90%


,법정동명,행정동명
0,가락동,3
3,가양동,4
6,갈현동,2
12,개봉동,3
13,개포동,5


In [ ]:
mapping = (
    mapping.groupby('법정동명')['행정동명']
    .first()
    .reset_index()
)

In [ ]:
#행정동 매핑
rent = rent.merge(
    mapping,
    on='법정동명',
    how='left'
)

print(
    "행정동 결측:",
    rent['행정동명'].isna().sum()
)

행정동 결측: 0


In [ ]:
num_cols = [
    '보증금(만원)',
    '임대료(만원)',
    '임대면적(㎡)'
]

for col in num_cols:
    rent[col] = pd.to_numeric(
        rent[col],
        errors='coerce'
    )

rent = rent.dropna(
    subset=[
        '행정동명',
        '보증금(만원)',
        '임대면적(㎡)'
    ]
)

rent.head()

,접수연도,자치구코드,자치구명,법정동코드,법정동명,지번구분,지번구분명,본번,부번,층,...,임대료(만원),건물명,건축년도,건물용도,계약기간,신규갱신여부,계약갱신권사용여부,종전 보증금,종전 임대료,행정동명
0,2026,11290,성북구,13400,길음동,1.0,대지,1283.0,0.0,6.0,...,200,길음뉴타운6단지(래미안),2007.0,아파트,26.08~28.08,신규,NaN,NaN,NaN,길음1동
1,2026,11680,강남구,11400,일원동,1.0,대지,743.0,0.0,9.0,...,450,디에이치자이개포,2021.0,아파트,26.07~28.07,신규,NaN,NaN,NaN,개포3동
2,2026,11560,영등포구,13300,대림동,1.0,대지,1126.0,0.0,12.0,...,71,HHouse대림뉴스테이,2017.0,아파트,26.06~27.06,갱신,○,10000.0,70.0,대림2동
3,2026,11305,강북구,10100,미아동,1.0,대지,791.0,339.0,5.0,...,90,제이스타빌(791-339),2015.0,연립다세대,26.06~28.06,신규,NaN,NaN,NaN,미아동
4,2026,11440,마포구,12700,상암동,1.0,대지,1734.0,0.0,3.0,...,68,상암 한화 오벨리스크,2013.0,오피스텔,26.05~27.05,신규,NaN,NaN,NaN,상암동


In [ ]:
#전/월세 분리
rent_deposit = rent[rent['전월세 구분'] == '전세']
rent_monthly = rent[rent['전월세 구분'] == '월세']

In [ ]:
#월세 데이터 확인
display(
    rent_monthly['임대료(만원)']
    .describe()
)

,임대료(만원)
count,153321.000000
mean,74.747569
std,84.005654
min,0.000000
25%,37.000000
50%,56.000000
75%,85.000000
max,4000.000000


In [ ]:
rent_deposit = (
    rent_deposit.groupby('행정동명')
    .agg({
        '보증금(만원)':'median',
        '임대면적(㎡)':'median'
    })
    .reset_index()
)

rent_monthly = (
    rent_monthly.groupby('행정동명')
    .agg({
        '보증금(만원)':'median',
        '임대료(만원)':'median',
        '임대면적(㎡)':'median'
    })
    .reset_index()
)

In [ ]:
rent_deposit.head()

,행정동명,보증금(만원),임대면적(㎡)
0,가락2동,65000.0,59.968
1,가리봉동,15000.0,30.330
2,가산동,20050.0,25.630
3,가양1동,34000.0,49.500
4,가회동,24750.0,43.480


In [ ]:
rent_monthly.head()

,행정동명,보증금(만원),임대료(만원),임대면적(㎡)
0,가락2동,14954.0,61.0,43.070
1,가리봉동,1000.0,47.0,22.500
2,가산동,1000.0,55.0,17.690
3,가양1동,3000.0,60.0,34.440
4,가회동,4000.0,67.5,43.625


In [ ]:
rent_deposit.to_csv('/content/drive/MyDrive/Dataset/Preprocessed/rent_deposit.csv', index=False)
rent_monthly.to_csv('/content/drive/MyDrive/Dataset/Preprocessed/rent_montly.csv', index=False)